# 18e — Refugia transform conditionality (E20 / PF-1): the one guarded run

S0 (balanced) at SSP585 with the refugia layer replaced by `log1p(1/v)` (`input_data/aligned_stack/e20_log_refugia_585.tif`) and the
weights re-derived under constant intended influence (18d, `spec/v3.1/e20_refugia_transform.json`); targets as registered. Anchor at
opt_gap 1e-4, then the guarded MGA (k = 50, g = 5%, per-block floors at 95% of the anchor's capture — the applied semantics) into
`runs_v3.1/e20_log/s0_ssp585_theta5/`. Resumable (skips when the guard tif exists). ~1 h. Live internet (Gurobi WLS).
Then 18g.

In [3]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
VERSION <- "v3.1"
MANIFEST_REL <- sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL <- sprintf("analyses/y2y/runs_%s", VERSION); EFG_SUBDIR_EXPECTED <- paste0("iucn_efg_", sub("\\..*$", "", VERSION))
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) == 12)
E20 <- jsonlite::read_json(file.path(PROJ, sprintf("analyses/y2y/spec/%s/e20_refugia_transform.json", VERSION)))
row <- MAN[MAN$formulation_id == E20$base_formulation, ]; stopifnot(nrow(row) == 1)
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))
BLOCKS <- lapply(ER$e17_t3$blocks, unlist); FLOOR_G <- as.numeric(E20$floor_g)
OUT_REL <- file.path(RUNS_REL, "e20_log", E20$base_formulation); OUT <- file.path(PROJ, OUT_REL); dir.create(OUT, recursive = TRUE, showWarnings = FALSE)
# the base context with the refugia layer swapped for the log1p(1/v) test layer BEFORE ingest (the 245-realization mechanism)
ctx <- pr_setup(mpath, PROJ)
ctx$layers$path[ctx$layers$name == "climate_type_macrorefugia"] <- E20$layer_patch[["climate_type_macrorefugia"]]
ctx <- modifyList(ctx, pr_ingest(ctx)); ctx <- modifyList(ctx, pr_planning_units(ctx))
stopifnot(all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx$layers$path[ctx$layers$role == "feature_efg"])))
# identity check on the patched features: ingest sum-normalizes every feature, so compare max/mean (scale-invariant) with 18d's record
for (nm in names(E20$layer_check)) {
  v <- terra::values(ctx$features[[nm]]); r_obs <- max(v, na.rm = TRUE) / mean(v, na.rm = TRUE); r_exp <- as.numeric(E20$layer_check[[nm]])
  if (!(abs(r_obs / r_exp - 1) < 0.02)) stop(sprintf("%s: the ingested feature is not the test layer (max/mean %.2f vs expected %.2f)", nm, r_obs, r_exp))
  cat(sprintf("   %s: test layer confirmed (max/mean %.2f)
", nm, r_obs))
}
w <- unlist(E20$weights); t <- unlist(E20$targets)
cat(sprintf("E20 %s: S0@SSP585 under %s | refugia weight %.4f (registered %.4f) | k %d g %.2f floor %.2f -> %s\n",
            E20$variant, E20$variant, w[["climate_type_macrorefugia"]], unlist(E20$weights_registered)[["climate_type_macrorefugia"]],
            E20$k, as.numeric(E20$band_g), FLOOR_G, OUT_REL))


manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 features (8 continuous + 20 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 28 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
   climate_type_macrorefugia: test layer confirmed (max/mean 7.31)
E20 log1p(1/v): S0@SSP585 under log1p(1/v) | refugia weight 1.6932 (registered 1.4600) | k 50 g 0.05 floor 0.05 -> analyses/y2y/runs_v3.1/e20_log/s0_ssp585_theta5


In [4]:
# ---- anchor + guarded MGA (mirrors 18's run_guard, without the frozen-anchor assert: this anchor is new) ----------------
tif <- file.path(OUT, "mga_guard_g05.tif")
if (file.exists(tif)) {
  cat("guard tif exists -- skipped (delete the folder to re-run)\n")
} else {
  actx <- pr_override(ctx, targets = as.list(t), feature_weight_multipliers = as.list(w),
                      results_dir = OUT_REL, results_subdir = "guard_build",
                      solver = "gurobi", decision_type = "binary", opt_gap = as.numeric(E20$opt_gap), portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  t0 <- proc.time()[["elapsed"]]
  anchor <- mga_anchor(cm, opt_gap = as.numeric(E20$opt_gap))
  write_layers <- function(M, path, prefix) {
    layers <- lapply(seq_len(nrow(M)), function(i) {
      r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(M[i, ]); terra::values(r) <- v; r })
    s <- terra::rast(layers); names(s) <- sprintf("%s_%02d", prefix, seq_len(nrow(M)))
    terra::writeRaster(s, path, overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  }
  write_layers(matrix(anchor$x, nrow = 1), file.path(OUT, "anchor.tif"), "anchor")
  jsonlite::write_json(list(formulation_id = E20$base_formulation, variant = E20$variant, layer_patch = E20$layer_patch,
                            anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime,
                            n_selected = sum(anchor$x), weights = as.list(w), targets = as.list(t),
                            created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(OUT, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  floors <- list(ctx = actx, blocks = BLOCKS, g = FLOOR_G)
  gen <- mga_generate(cm, anchor, g = as.numeric(E20$band_g), k = as.integer(E20$k), floors = floors)
  write_layers(gen$members, tif, "guard")
  write.csv(gen$certificates, file.path(OUT, "certificates_guard.csv"), row.names = FALSE)
  jsonlite::write_json(list(formulation_id = E20$base_formulation, kind = "mga", variant = E20$variant, floor_g = FLOOR_G, blocks = BLOCKS,
                            band_g = as.numeric(E20$band_g), k = gen$k, anchor_objective_resolved = anchor$z,
                            total_runtime_s = sum(gen$certificates$runtime_s), wall_s = proc.time()[["elapsed"]] - t0,
                            created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(OUT, "guard_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  cat(sprintf("E20 guarded run complete: anchor %.6f | %d members | %.0f s solver / %.0f s wall -> %s\n",
              anchor$z, gen$k, sum(gen$certificates$runtime_s), proc.time()[["elapsed"]] - t0, OUT_REL))
}
cat("next: 18g_e20_analysis\n")


guard tif exists -- skipped (delete the folder to re-run)
next: 18f_e20_analysis
